# TabPFN → drzewo decyzyjne (rozwiązanie 2)

Colab: **Runtime → Change runtime type → T4 GPU**, potem Run all.

TabPFN uczy się z `val.csv`, etykietuje `train.csv`, a sklearn-drzewo destyluje te decyzje na nazwanych sygnaturach akustycznych. Aplikacja warsztatowa (`app_tabpfn.py`) liczy już tylko drzewo — CPU, ścieżka if/then, punkty za explainability.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q tabpfn scikit-learn pandas numpy torch joblib plotly
    from google.colab import files
    print("Wgraj val.csv, train.csv, test.csv oraz tabpfn_diagnose.py")
    files.upload()

import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 762.1/762.1 kB 20.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.5 MB/s eta 0:00:00
Wgraj val.csv, train.csv, test.csv oraz tabpfn_diagnose.py


In [ ]:
from tabpfn_diagnose import TabPFNTreeDiagnoser, pick_device
import pandas as pd

val = pd.read_csv("val.csv")
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
print(val.shape, train.shape, test.shape, "device=", pick_device())

## Nauczyciel + destylacja

Na T4 włącz `--cv` w komórce poniżej (GroupKFold po silniku). Na CPU zostaw `do_cv=False`.

In [ ]:
device = pick_device()
n_estimators = 8 if device == "cuda" else 4
do_cv = device == "cuda"  # T4: uczciwy score; CPU: pomiń

model = TabPFNTreeDiagnoser().fit(
    val,
    train,
    device=device,
    n_estimators=n_estimators,
    do_cv=do_cv,
)
model.save()
print(model.meta)

## Drzewo, które zobaczy mechanik

In [ ]:
print(model.rules_text())

## Submit + zgodność nauczyciel / student

In [ ]:
sub_tree = model.predict(test)
sub_tabpfn = model.predict_teacher(test, model.teacher_)
sub_tree.to_csv("predictions_tree.csv", index=False)
sub_tabpfn.to_csv("predictions_tabpfn.csv", index=False)
agree = (sub_tree["label"] == sub_tabpfn["label"]).mean()
print(f"zgoda drzewo vs TabPFN na teście: {agree:.3f}")
print("TabPFN\n", sub_tabpfn["label"].value_counts())
print("drzewo\n", sub_tree["label"].value_counts())

if IN_COLAB:
    files.download("predictions_tabpfn.csv")
    files.download("predictions_tree.csv")
    files.download("artifacts/diagnoser_tree.joblib")

Lokalnie po pobraniu `diagnoser_tree.joblib` do `artifacts/`:

```bash
streamlit run app_tabpfn.py
```